# 01 - Data Acquisition 

#Loading the RA microbiome dataset (Li, Jing; Xu, Jun (2024))

#Source: https://figshare.com/articles/dataset/Data_for_publication_in_Scientific_Data/27603876/2

#2,238 individuals: 1,034 RA patients + 1,204 healthy controls.

#Outputs saved to data/processed/:
# - feature_table.csv: samples × ASVs (raw counts)
# - metadata.csv: clinical variables per sample
# - taxonomy.csv:ASV → phylum/genus mapping

In [1]:
import pyreadr
import pandas as pd
import numpy as np
import sys
import os

sys.path.insert(0, os.path.abspath('..'))
from config import COHORTS, PARAMS, PATHS

RAW = "../data/raw/"
PROCESSED = "../data/processed/"

print("Libraries loaded ✅")

Libraries loaded ✅


## Step 1 — Load the raw .rds files

#pyreadr.read_r() reads R data files directly into Python.
#It returns a dictionary — we grab the first value which is our DataFrame.

In [4]:
# Load all three files
asv_raw  = pyreadr.read_r(RAW + "1.16S.ASV.profile.original.rds")
meta_raw = pyreadr.read_r(RAW + "1.sample.info.original.rds")
tax_raw  = pyreadr.read_r(RAW + "1.taxonomy.info.rds")

# pyreadr returns an OrderedDict — grab the first (only) value
asv_df  = list(asv_raw.values())[0]
meta_df = list(meta_raw.values())[0]
tax_df  = list(tax_raw.values())[0]

print("=== ASV TABLE ===")
print(f"Shape: {asv_df.shape}")
print(asv_df.iloc[:3, :5])

print("\n=== METADATA ===")
print(f"Shape: {meta_df.shape}")
print(meta_df.head(3))

print("\n=== TAXONOMY ===")
print(f"Shape: {tax_df.shape}")
print(tax_df.head(3))

=== ASV TABLE ===
Shape: (39868, 2238)
                                  HC0017  HC0018  HC0021  HC0026  HC0027
rownames                                                                
88f476b4cfd899a60e21886d0f3bfb0e      32     141     332       0    1287
055095f4a78cd2559cf33874d95715de      20       0    1215       0     106
959c70f2ef9414a19f279f494ea4f305     110       0     271       0      30

=== METADATA ===
Shape: (2238, 5)
          Collection date Stool characteristics SampleID Group (1=HC,2=RA)  \
rownames                                                                     
HC0017            2015.12                Normal   HC0017                 1   
HC0018            2015.12                Normal   HC0018                 1   
HC0021            2015.12                Normal   HC0021                 1   

         Location  
rownames           
HC0017    Beijing  
HC0018    Beijing  
HC0021    Beijing  

=== TAXONOMY ===
Shape: (40597, 2)
                                  

In [5]:
## Step 2 — Transpose ASV table

#Currently: rows = ASVs, columns = samples.
#We need: rows = samples, columns = ASVs.
#This is the standard orientation for microbiome analysis.

# Transpose so rows = samples, columns = ASVs
asv_df = asv_df.T

# The index (row names) are now sample IDs — name it clearly
asv_df.index.name = "sample_id"

# Convert all values to integers (counts should never be floats)
asv_df = asv_df.astype(int)

print(f"ASV table after transpose: {asv_df.shape}")
print(f"  → {asv_df.shape[0]} samples × {asv_df.shape[1]} ASVs")
print(f"\nFirst 3 samples, first 4 ASVs:")
print(asv_df.iloc[:3, :4])

print(f"\nTotal read counts per sample (first 5):")
print(asv_df.sum(axis=1).head())

ASV table after transpose: (2238, 39868)
  → 2238 samples × 39868 ASVs

First 3 samples, first 4 ASVs:
rownames   88f476b4cfd899a60e21886d0f3bfb0e  055095f4a78cd2559cf33874d95715de  \
sample_id                                                                       
HC0017                                   32                                20   
HC0018                                  141                                 0   
HC0021                                  332                              1215   

rownames   959c70f2ef9414a19f279f494ea4f305  4701033a29524afa86c12528501542ad  
sample_id                                                                      
HC0017                                  110                               180  
HC0018                                    0                                55  
HC0021                                  271                                 0  

Total read counts per sample (first 5):
sample_id
HC0017    10000
HC0018    10000
HC0021  

In [6]:
## Step 3 — Clean metadata

#Rename columns to be code-friendly (no spaces).
#Convert Group from numeric (1/2) to readable labels (HC/RA).

# Rename columns to clean, code-friendly names
meta_df.index.name = "sample_id"

meta_df = meta_df.rename(columns={
    "Collection date":      "collection_date",
    "Stool characteristics": "stool_type",
    "SampleID":             "sample_id_col",
    "Group (1=HC,2=RA)":   "group_numeric",
    "Location":             "location"
})

# Convert numeric group to readable label
meta_df["disease_status"] = meta_df["group_numeric"].map({1: "HC", 2: "RA"})

print("Metadata columns:", meta_df.columns.tolist())
print(f"\nDisease status counts:")
print(meta_df["disease_status"].value_counts())
print(f"\nLocations:")
print(meta_df["location"].value_counts())
print(f"\nPreview:")
print(meta_df[["disease_status", "collection_date", 
               "stool_type", "location"]].head(5))

Metadata columns: ['collection_date', 'stool_type', 'sample_id_col', 'group_numeric', 'location', 'disease_status']

Disease status counts:
Series([], Name: count, dtype: int64)

Locations:
location
Beijing      1928
Shanxi        129
Henan         105
Hubei          47
Guangdong      29
Name: count, dtype: int64

Preview:
          disease_status  collection_date stool_type location
sample_id                                                    
HC0017               NaN          2015.12     Normal  Beijing
HC0018               NaN          2015.12     Normal  Beijing
HC0021               NaN          2015.12     Normal  Beijing
HC0026               NaN          2015.12     Normal  Beijing
HC0027               NaN          2015.12     Normal  Beijing


In [7]:
## Step 4 — Parse taxonomy strings

#The Taxon column contains full lineage strings like:
#  k__Bacteria;p__Proteobacteria;c__Gammaproteobacteria;o__...;g__Genus

#We split this into one column per taxonomic level.
#This lets us later roll up ASV counts to Genus or Family level easily. 

def parse_taxonomy(taxon_string):
    """
    Parse a QIIME2/Greengenes taxonomy string into a dictionary.
    
    Input:  "k__Bacteria;p__Proteobacteria;c__Gammaproteobacteria;..."
    Output: {"Kingdom": "Bacteria", "Phylum": "Proteobacteria", ...}
    """
    # Map the single-letter prefix to full level name
    prefix_map = {
        "k": "Kingdom",
        "p": "Phylum", 
        "c": "Class",
        "o": "Order",
        "f": "Family",
        "g": "Genus",
        "s": "Species"
    }
    
    result = {v: "Unknown" for v in prefix_map.values()}
    
    if not isinstance(taxon_string, str):
        return result
    
    parts = taxon_string.split(";")
    for part in parts:
        part = part.strip()
        if "__" in part:
            prefix, value = part.split("__", 1)
            prefix = prefix.strip()
            value = value.strip()
            if prefix in prefix_map and value != "":
                result[prefix_map[prefix]] = value
    
    return result

# Apply to every row in taxonomy table
parsed = tax_df["Taxon"].apply(parse_taxonomy)
tax_parsed = pd.DataFrame(parsed.tolist(), index=tax_df.index)
tax_parsed.index.name = "asv_id"

# Add confidence score
tax_parsed["confidence"] = tax_df["Confidence"].values

print(f"Taxonomy table shape: {tax_parsed.shape}")
print(f"\nPhylum breakdown (top 10):")
print(tax_parsed["Phylum"].value_counts().head(10))
print(f"\nGenus breakdown (top 10):")
print(tax_parsed["Genus"].value_counts().head(10))
print(f"\nPreview:")
print(tax_parsed[["Phylum","Family","Genus"]].head(5))

Taxonomy table shape: (40597, 8)

Phylum breakdown (top 10):
Phylum
Firmicutes         29217
Bacteroidetes       6039
Proteobacteria      2854
Actinobacteria      1374
Unknown              275
Tenericutes          274
Fusobacteria         269
Patescibacteria      143
Cyanobacteria         84
Synergistetes         26
Name: count, dtype: int64

Genus breakdown (top 10):
Genus
Unknown                    5018
Bacteroides                2941
Faecalibacterium           1526
Lachnospiraceae UCG-004    1228
Dialister                   988
Prevotella 9                941
Escherichia-Shigella        885
Roseburia                   871
uncultured                  832
Blautia                     738
Name: count, dtype: int64

Preview:
                                          Phylum              Family  \
asv_id                                                                 
88f476b4cfd899a60e21886d0f3bfb0e  Proteobacteria  Enterobacteriaceae   
055095f4a78cd2559cf33874d95715de   Bacteroidetes   

In [8]:
## Step 5 — Align indices and verify consistency

#Make sure sample IDs in the ASV table and metadata match exactly.
#Drop any ASVs from taxonomy that aren't in the ASV table and vice versa.

# ── Check sample alignment ────────────────────────────────────────────────
asv_samples  = set(asv_df.index)
meta_samples = set(meta_df.index)

in_both     = asv_samples & meta_samples
only_in_asv = asv_samples - meta_samples
only_in_meta = meta_samples - asv_samples

print(f"Samples in ASV table:  {len(asv_samples)}")
print(f"Samples in metadata:   {len(meta_samples)}")
print(f"Samples in both:       {len(in_both)}")
print(f"Only in ASV (no meta): {len(only_in_asv)}")
print(f"Only in meta (no ASV): {len(only_in_meta)}")

# Keep only samples present in both
shared_samples = sorted(list(in_both))
asv_df  = asv_df.loc[shared_samples]
meta_df = meta_df.loc[shared_samples]

# ── Check ASV alignment between table and taxonomy ─────────────────────
asv_ids_table = set(asv_df.columns)
asv_ids_tax   = set(tax_parsed.index)

print(f"\nASVs in feature table:  {len(asv_ids_table)}")
print(f"ASVs in taxonomy:       {len(asv_ids_tax)}")
print(f"ASVs in both:           {len(asv_ids_table & asv_ids_tax)}")

# Keep only ASVs that have taxonomy annotations
shared_asvs = sorted(list(asv_ids_table & asv_ids_tax))
asv_df      = asv_df[shared_asvs]
tax_parsed  = tax_parsed.loc[shared_asvs]

print(f"\nAfter alignment:")
print(f"  ASV table:  {asv_df.shape}")
print(f"  Metadata:   {meta_df.shape}")
print(f"  Taxonomy:   {tax_parsed.shape}")

Samples in ASV table:  2238
Samples in metadata:   2238
Samples in both:       2238
Only in ASV (no meta): 0
Only in meta (no ASV): 0

ASVs in feature table:  39868
ASVs in taxonomy:       40597
ASVs in both:           39868

After alignment:
  ASV table:  (2238, 39868)
  Metadata:   (2238, 6)
  Taxonomy:   (39868, 8)


In [11]:
## Step 6 — Save to data/processed/

#These three files are the input for every subsequent notebook.

asv_df.to_csv(PROCESSED + "feature_table.csv")
meta_df.to_csv(PROCESSED + "metadata.csv")
tax_parsed.to_csv(PROCESSED + "taxonomy.csv")

print("Saved:")
print(f"  ✅ feature_table.csv  {asv_df.shape}")
print(f"  ✅ metadata.csv       {meta_df.shape}")
print(f"  ✅ taxonomy.csv       {tax_parsed.shape}")

# Final sanity check
print("\n=== FINAL SANITY CHECK ===")
print(f"RA samples:  {(meta_df['disease_status']=='RA').sum()}")
print(f"HC samples:  {(meta_df['disease_status']=='HC').sum()}")
print(f"Total ASVs:  {asv_df.shape[1]}")
print(f"Genera in taxonomy: {tax_parsed['Genus'].nunique()}")
print(f"Phyla in taxonomy:  {tax_parsed['Phylum'].nunique()}")
print("\n✅ Data acquisition complete.")

Saved:
  ✅ feature_table.csv  (2238, 39868)
  ✅ metadata.csv       (2238, 6)
  ✅ taxonomy.csv       (39868, 8)

=== FINAL SANITY CHECK ===
RA samples:  0
HC samples:  0
Total ASVs:  39868
Genera in taxonomy: 447
Phyla in taxonomy:  15

✅ Data acquisition complete.


In [12]:
# group_numeric holds strings (1.0, 2.0), so .map({1:..., 2:...}) found no matches.
# Cast to int first, then map.

print("group_numeric dtype:", meta_df["group_numeric"].dtype)
print(meta_df["group_numeric"].head())

meta_df["disease_status"] = (
    meta_df["group_numeric"]
    .astype(float)
    .astype(int)
    .map({1: "HC", 2: "RA"})
)

print("\nAfter fix:")
print(meta_df["disease_status"].value_counts())
print("NaN remaining:", meta_df["disease_status"].isna().sum())

# Overwrite the saved file with the corrected version
meta_df.to_csv(PROCESSED + "metadata.csv")
print("\nmetadata.csv updated")

group_numeric dtype: object
sample_id
HC0017    1
HC0018    1
HC0021    1
HC0026    1
HC0027    1
Name: group_numeric, dtype: object

After fix:
disease_status
HC    1204
RA    1034
Name: count, dtype: int64
NaN remaining: 0

metadata.csv updated
